In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [2]:
df=pd.read_csv(r"pakistan_tech_jobs_final.csv")

In [3]:
df.drop('job_description',inplace=True,axis=1)

In [4]:
df.isna().sum()
df['education']=df['education'].fillna(df['education'].mode()[0])

In [5]:
#K means to cluster job categories
#Linear to predict min max salries 
#logistic to predict job_title
#KNN to predict five related jobs
#firstly preprocessing
df['job_title']=df['job_title'].fillna(df['job_title'].mode()[0])
df.isna().sum()
df.duplicated().sum()#No Duplicates

np.int64(0)

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
scaler=StandardScaler()

In [7]:
from sklearn.metrics import r2_score, mean_absolute_error

In [8]:
cols_for_sal=['experience_min','city','job_category','work_mode','education','experience_max','salary_min','salary_max']
reg_df=df[cols_for_sal]

In [9]:
reg_df['avg_experience']=((reg_df['experience_min']+reg_df['experience_max'])/2)
reg_df['avg_sal']=((reg_df['salary_min']+reg_df['salary_max'])/2)
reg_df.head()

,experience_min,city,job_category,work_mode,education,experience_max,salary_min,salary_max,avg_experience,avg_sal
0,5.0,Karachi,Data Science / AI / ML,Remote,Bachelor's,8.0,560000.0,840000.0,6.5,700000.0
1,2.0,Lahore,Web Development,On-site,Bachelor's,5.0,80000.0,110000.0,3.5,95000.0
2,2.0,Lahore,Data Science / AI / ML,On-site,Bachelor's,5.0,60000.0,100000.0,3.5,80000.0
3,2.0,Karachi,Web Development,On-site,Bachelor's,5.0,50000.0,60000.0,3.5,55000.0
4,0.0,Karachi,Mobile Development,Remote,Bachelor's,2.0,40000.0,59998.0,1.0,49999.0


In [10]:
top_cities = ['Karachi', 'Lahore', 'Islamabad', 'Rawalpindi']
reg_df['city_tier'] = reg_df['city'].apply(lambda x: x if x in top_cities else 'Other')

In [11]:
reg_df=pd.get_dummies(reg_df,drop_first=True,columns=['work_mode','city_tier','job_category'],dtype=int)
reg_df['education']=reg_df['education'].replace({"Master's":1,"Bachelor's":0}).astype(int)

In [12]:
reg_df.drop('city',axis=1,inplace=True)
reg_df.head()

,experience_min,education,experience_max,salary_min,salary_max,avg_experience,avg_sal,work_mode_On-site,work_mode_Remote,city_tier_Karachi,city_tier_Lahore,city_tier_Other,city_tier_Rawalpindi,job_category_Cybersecurity,job_category_Data Science / AI / ML,job_category_Mobile Development,job_category_QA / Testing,job_category_Software Development,job_category_Web Development
0,5.0,0,8.0,560000.0,840000.0,6.5,700000.0,0,1,1,0,0,0,0,1,0,0,0,0
1,2.0,0,5.0,80000.0,110000.0,3.5,95000.0,1,0,0,1,0,0,0,0,0,0,0,1
2,2.0,0,5.0,60000.0,100000.0,3.5,80000.0,1,0,0,1,0,0,0,1,0,0,0,0
3,2.0,0,5.0,50000.0,60000.0,3.5,55000.0,1,0,1,0,0,0,0,0,0,0,0,1
4,0.0,0,2.0,40000.0,59998.0,1.0,49999.0,0,1,1,0,0,0,0,0,1,0,0,0


In [13]:
senior_kw = ['senior', 'sr', 'lead', 'principal', 'architect', 'head', 'manager', 'director']
reg_df['is_senior'] = df['job_title'].fillna('').str.lower().apply(lambda t: int(any(k in t for k in senior_kw)))
reg_df.head()

,experience_min,education,experience_max,salary_min,salary_max,avg_experience,avg_sal,work_mode_On-site,work_mode_Remote,city_tier_Karachi,city_tier_Lahore,city_tier_Other,city_tier_Rawalpindi,job_category_Cybersecurity,job_category_Data Science / AI / ML,job_category_Mobile Development,job_category_QA / Testing,job_category_Software Development,job_category_Web Development,is_senior
0,5.0,0,8.0,560000.0,840000.0,6.5,700000.0,0,1,1,0,0,0,0,1,0,0,0,0,1
1,2.0,0,5.0,80000.0,110000.0,3.5,95000.0,1,0,0,1,0,0,0,0,0,0,0,1,0
2,2.0,0,5.0,60000.0,100000.0,3.5,80000.0,1,0,0,1,0,0,0,1,0,0,0,0,0
3,2.0,0,5.0,50000.0,60000.0,3.5,55000.0,1,0,1,0,0,0,0,0,0,0,0,1,0
4,0.0,0,2.0,40000.0,59998.0,1.0,49999.0,0,1,1,0,0,0,0,0,1,0,0,0,0


In [14]:
# Outlier handling for salary_max (top 2%)
cap_max = reg_df['salary_max'].quantile(0.98)
reg_df_max = reg_df[reg_df['salary_max'] <= cap_max].copy()
# Features and target for salary_max (dropping other salary targets and min experience)
X_max = reg_df_max.drop(columns=['salary_min', 'salary_max', 'avg_sal', 'experience_min', 'avg_experience'], errors='ignore')
y_max = reg_df_max['salary_max']

In [15]:
X_train,X_test,y_train,y_test = train_test_split(X_max, y_max, test_size=0.25, random_state=42)
model_max = LinearRegression()
model_max.fit(X_train, y_train)
y_pred = model_max.predict(X_test)
y_pred_train = model_max.predict(X_train)
r2 = r2_score(y_test, y_pred)
print(f"Test R2: {r2}")
print(f"Train R2: {r2_score(y_train, y_pred_train)}")
print(f"MAE: {mean_absolute_error(y_test, y_pred)}")

Test R2: 0.7764678069934021
Train R2: 0.7201959127013733
MAE: 21300.620075326933


In [17]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(
    model_max,
    X_train,
    y_train,
    cv=5,
    scoring='r2'
)

print("CV scores:", scores)
print("Average CV R2:", scores.mean())

CV scores: [0.75020823 0.71833301 0.68194425 0.70912255 0.69875265]
Average CV R2: 0.7116721382033399
